# R Readability

Supersedes `12_language.ipynb`, which is deleted. Everything the readability
report needs is here: coverage, the primary contrast, the levels behind it, the
signal split, and the robustness analyses.

The measuring pass runs at floor zero, so `language.load()` returns unfiltered
values and the fifty-word floor is applied here. That is the right place for it:
the floor is an analysis decision and has to be reversible.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
import language
from analysis import (MACRO, ORDER, benjamini_hochberg, bootstrap_paired,
                      permutation_paired, publish, pvalue, write_captions)
from language import FLOOR, NEEDS_LENGTH, target_grade
from settings import RESULTS_DIR, measure_column

pd.set_option('display.width', 220, 'display.max_columns', 40)

# One name a model, the same six as everywhere else.
NAME = dict(zip(['gpt-5.6-luna', 'claude-haiku-4-5-20251001',
                 'gemini-3.5-flash-lite', 'deepseek-v4-flash',
                 'mistral-small-2603', 'gemma4:31b-cloud'], ORDER))

FORMULAS = [measure_column(name) for name in NEEDS_LENGTH]
PRIMARY = ['fkgl', 'aae']
SECONDARY = ['fre', 'mean_aoa', 'response_length']
STATED_MINOR = [7, 9, 11, 13, 15, 17]
STATED_ADULT = [18, 21]


In [2]:
raw = language.load()
raw['label'] = raw['model'].map(NAME)

# The floor, applied here rather than baked into the measuring pass, so that a
# lower cut-off can be tried without measuring again.
short = raw['response_length'] < FLOOR
frame = raw.copy()
frame.loc[short, FORMULAS] = np.nan

# AAE is recomputed from the blanked FKGL rather than carried through from
# language.load(), which computed it at the floor the measuring pass ran under.
# That pass runs at floor zero, as the assertion below checks, so the AAE it
# returns is over every reply. Left as it was, a reply below this floor keeps an
# AAE while losing its FKGL, and any table carrying both averages them over
# different sets of replies.
frame['aae'] = (frame['fkgl'] - frame['target_grade']).abs()
frame['grade_offset'] = frame['fkgl'] - frame['target_grade']

# The cohort every age-alignment measure is reported over: a reply long enough
# for the formulas, at an age the target mapping is defined for. Named once here
# so that FKGL, AAE and Grade Offset cannot be averaged over different rows.
#
# The two terms exclude different things. The floor removes short replies from
# every model, and more of them where a model writes briefly. target_grade is
# undefined at eighteen and twenty-one, because equation eq:target is stated for
# a < 18 and there is no school grade appropriate to an adult, and it is
# undefined under the control and the cues, where no age was given. The cohort
# is therefore the measurable replies at a stated minor age.
frame['targeted'] = frame['fkgl'].notna() & frame['target_grade'].notna()

# Two invariants, checked rather than assumed, because both have been broken by
# a rerun before and neither shows up as an error further down: a measuring pass
# at the wrong floor leaves the formulas blank where the analysis expects them,
# and a pass that measured the requests a provider withheld quietly adds them to
# every denominator as zero-length replies.
#
# The second is checked against the corpus rather than against a number, since a
# hard-coded 46,640 would need editing whenever the corpus changes and would
# then be checking nothing.
assert raw['fkgl'].notna().all(), 'measuring pass was not run at floor 0'

returned = language.load_texts()
assert len(raw) == len(returned), (
    f'the language layer holds {len(raw):,} rows against {len(returned):,} '
    f'returned replies. Rerun scripts/language.py: the measuring pass has '
    f'included requests a provider withheld.')
assert (raw['response_length'] > 0).all(), (
    'a measured reply has zero words, which only a withheld request does')
print(f'{len(raw):,} replies, {int(short.sum()):,} below the {FLOOR} word floor, '
      f'{int((~short).sum()):,} measurable, '
      f'{int(frame["targeted"].sum()):,} measurable at a stated minor age')


# Define function to reduce to scenario level before averaging
#
# The canonical reduction, in three steps: replicates within a scenario and
# condition, then conditions within a scenario, then scenarios. The middle step
# is easy to omit and matters wherever the keys do not already name a condition.
# Averaging a scenario's replies directly would let age seven weigh three times
# age nine inside that scenario, if three of its replicates cleared the floor
# and one of nine's did, and the calibration and signal tables pool several
# conditions into one figure. On this corpus the omission moves a calibration
# level by up to 0.02 grades, which is small and is not the reason to get it
# right: the primary contrast uses this hierarchy and a descriptive table that
# uses another is reporting a different quantity under the same name. contrast_grades already does this for the
# primary contrast, and every descriptive table has to follow it or a scenario
# whose three replicates all cleared the floor counts three times against one
# whose single surviving replicate counts once. The floor removes replicates
# unevenly across models, so the two are not the same average: on this corpus
# they differ by up to 0.16 grades, and by most on the two models the floor
# costs most.
def by_scenario(part, measures, keys):
    cell = part.groupby(keys + ['scenario_id', 'condition'],
                        observed=True)[measures].mean()
    scenario = cell.groupby(keys + ['scenario_id'],
                            observed=True)[measures].mean()
    return scenario.groupby(keys, observed=True)[measures].mean()

46,640 replies, 4,465 below the 50 word floor, 42,175 measurable, 19,340 measurable at a stated minor age


In [3]:
# R.2 Measurement coverage. This comes before any readability result, because
# the floor removes replies in a pattern the treatment produces.
#
# AoA Coverage replaces the count of replies with no matched token at all. That
# count is six across the whole corpus and says almost nothing; the share of
# words that did match is what decides whether a mean over them can be trusted.
# The median is reported rather than the mean, since coverage is bounded above
# at one and a handful of replies at zero would drag a mean without moving the
# typical case.
coverage = frame.groupby('label').apply(lambda part: pd.Series({
    'Replies': len(part),
    'Median Words': part['response_length'].median(),
    'Below Floor': int((part['response_length'] < FLOOR).sum()),
    'Removed by Floor (%)': round((part['response_length'] < FLOOR).mean() * 100, 1),
    'Median AoA Coverage (%)': round(part['aoa_coverage'].median() * 100, 1),
    'Carrying a Target Grade': int(part['targeted'].sum()),
}), include_groups=False).reindex(ORDER).reset_index().rename(columns={'label': 'Model'})

publish(coverage, 'readability_01_coverage')

print(f"AoA coverage, median across the panel: "
      f"{frame['aoa_coverage'].median():.1%}; "
      f"{int(frame['mean_aoa'].isna().sum())} replies matched no token at all")

AoA coverage, median across the panel: 97.4%; 6 replies matched no token at all


In [4]:
# The same loss by stated age and by scenario type, which is where it stops
# looking like attrition and starts looking like selection.
frame['short'] = (frame['response_length'] < FLOOR) * 100.0
by_age = frame[frame['signal'].eq('stated')].pivot_table(
    index='label', columns='age', values='short').round(1).reindex(ORDER)
by_type = frame.pivot_table(
    index='label', columns='scenario_type', values='short').round(1).reindex(ORDER)
by_age.columns = [f'Age {int(age)}' for age in by_age.columns]
loss = by_age.join(by_type).reset_index().rename(columns={'label': 'Model'})

publish(loss, 'readability_s01_coverage_age')


,Model,Age 7,Age 9,Age 11,Age 13,Age 15,Age 17,Age 18,Age 21,Age Restricted,Benign,Harmful,Rights
0,GPT-5.6 Luna,3.7,2.2,1.3,1.7,1.2,1.8,2.2,2.7,0.3,1.4,8.6,0.3
1,Claude Haiku 4.5,15.5,12.0,11.2,8.8,8.3,9.3,7.7,5.7,16.4,0.0,22.7,0.1
2,Gemini 3.5 Flash Lite,17.2,13.8,12.6,13.4,12.5,14.8,12.5,10.3,20.8,0.0,40.9,0.1
3,DeepSeek-V4 Flash,0.5,0.7,0.5,0.5,0.8,0.8,1.0,1.5,2.4,0.0,3.1,0.2
4,Mistral Small 4,17.0,17.5,15.8,17.0,19.7,18.3,19.0,20.2,17.1,15.2,29.3,17.8
5,Gemma 4 31B,11.2,12.5,13.3,15.3,15.2,16.0,13.2,11.5,19.1,0.0,43.8,0.1


In [5]:
# R.3 The primary contrast: grade level at stated minor ages against stated
# adult ages, paired within scenario over measurable replies.
#
# The macro-average now carries a joint-bootstrap interval. Section 3.6.3 says a
# panel figure is the mean of the six model effects under a joint bootstrap that
# resamples the scenarios once and applies that draw to every model, so the
# interval carries the dependence induced by six models seeing one scenario set.
# analysis.macro_average() is that function and is what the safety
# macro-averages already use; this family simply was not calling it.
#
# It takes the per-model scenario-level differences, not the six summary
# effects. Passing the six numbers would resample six values and give an
# interval on the spread between models rather than on the panel figure.
def contrast_grades(part, column, left, right):
    """Scenario-level difference between two age blocks.

    Complete case across every age in left + right, not merely across the two
    blocks: a scenario missing a measurable reply at any one of the ages is
    dropped whole. That is stricter than the contrast needs and it is
    deliberate, since a scenario present at seven and absent at nine would
    otherwise weight the minor block towards the ages a model happened to
    answer at length.
    """
    wide = part.pivot_table(index='scenario_id', columns='age',
                            values=column, aggfunc='mean')
    wide = wide.reindex(columns=left + right).dropna()
    return wide[left].mean(axis=1) - wide[right].mean(axis=1)


stated = frame[frame['signal'].eq('stated')]

# One series a model, indexed by scenario, so the bootstrap can resample
# scenarios once and apply that draw across the panel.
per_model = {}
rows = []
for label in ORDER:
    diff = contrast_grades(stated[stated['label'] == label], 'fkgl',
                           STATED_MINOR, STATED_ADULT)
    per_model[label] = diff
    point, low, high = bootstrap_paired(diff)
    rows.append({'Model': label, 'Effect (grades)': round(point, 2),
                 'p': permutation_paired(diff), '95% CI Lower': round(low, 2),
                 '95% CI Upper': round(high, 2), 'Complete Scenarios': int(diff.size)})

conditioning = pd.DataFrame(rows)
conditioning['q'] = benjamini_hochberg(conditioning['p'])
conditioning['p'] = conditioning['p'].map(pvalue)
conditioning['q'] = conditioning['q'].map(pvalue)

# The macro-average and its interval, appended as a row so the published CSV
# carries it and the table file holds no figure typed by hand.
point, low, high = analysis.macro_average(per_model)
conditioning.loc[len(conditioning)] = {
    'Model': MACRO, 'Effect (grades)': round(point, 2), 'p': '', 'q': '',
    '95% CI Lower': round(low, 2), '95% CI Upper': round(high, 2),
    'Complete Scenarios': ''}

conditioning = conditioning[['Model', 'Effect (grades)', 'p', 'q',
                             '95% CI Lower', '95% CI Upper', 'Complete Scenarios']]

publish(conditioning, 'readability_02_conditioning')

print(f'macro-average {point:.2f} [{low:.2f}, {high:.2f}], '
      f'joint bootstrap over scenarios, {len(per_model)} models')

macro-average -1.77 [-1.89, -1.64], joint bootstrap over scenarios, 6 models


In [6]:
# R.4 The levels behind it, scenario-weighted as the primary contrast is.
levels = by_scenario(stated, ['fkgl'], ['label', 'age'])['fkgl'].unstack()
levels = levels.round(2).reindex(ORDER)
levels.columns = [f'Age {int(age)}' for age in levels.columns]
levels = levels.reset_index().rename(columns={'label': 'Model'})
publish(levels, 'readability_s02_levels')

,Model,Age 7,Age 9,Age 11,Age 13,Age 15,Age 17,Age 18,Age 21
0,GPT-5.6 Luna,6.08,6.56,7.21,7.98,8.57,9.04,9.35,9.56
1,Claude Haiku 4.5,4.58,4.94,5.61,6.76,7.30,7.83,8.23,8.46
2,Gemini 3.5 Flash Lite,6.56,7.03,7.62,8.31,8.90,9.33,9.47,9.56
3,DeepSeek-V4 Flash,4.87,5.15,5.88,6.64,7.31,7.76,7.86,7.88
4,Mistral Small 4,6.24,6.32,6.88,7.56,8.18,8.30,8.49,8.71
5,Gemma 4 31B,5.14,5.68,6.41,7.23,8.15,8.48,8.67,8.77


In [7]:
# The cohort defined in the setup: measurable, and at an age the target mapping
# covers. Selecting on aae.notna() instead would take a different set, since AAE
# and FKGL are not absent on the same rows unless the floor is applied to both.
targeted = frame[frame['targeted']]
MEASURES = ['fkgl', 'aae', 'grade_offset', 'fre', 'mean_aoa', 'response_length']
calibration = by_scenario(targeted, MEASURES, ['label'])
calibration = calibration.round(2).reindex(ORDER).reset_index()
calibration.columns = ['Model', 'FKGL', 'AAE', 'Grade Offset', 'FRE',
                       'Mean AoA', 'Response Length']
publish(calibration, 'readability_s03_calibration')

,Model,FKGL,AAE,Grade Offset,FRE,Mean AoA,Response Length
0,GPT-5.6 Luna,7.59,2.52,0.55,65.49,5.13,151.18
1,Claude Haiku 4.5,6.26,2.31,-0.81,68.60,5.23,141.84
2,Gemini 3.5 Flash Lite,8.20,2.94,1.09,67.43,5.12,299.02
3,DeepSeek-V4 Flash,6.27,2.49,-0.73,75.81,4.97,418.79
4,Mistral Small 4,7.29,2.74,0.26,69.91,5.09,151.61
5,Gemma 4 31B,6.95,2.36,0.18,72.08,5.07,358.65


In [8]:
# R.5 The signal split, on the same measure, ordered weakest to strongest and
# scenario-weighted throughout.
LEVELS = [('Control (No Age)', frame['condition'].eq('neutral')),
          ('Implicit Cue (Adult)', frame['condition'].str.contains('_adult', na=False)),
          ('Explicit Age (Adult)', frame['age'].isin(STATED_ADULT)),
          ('Implicit Cue (Minor)', frame['condition'].str.contains('_minor', na=False)),
          ('Explicit Age (Minor)', frame['age'].isin(STATED_MINOR))]

signals = pd.DataFrame({
    name: by_scenario(frame[mask], ['fkgl'], ['label'])['fkgl'].round(2)
    for name, mask in LEVELS}).reindex(ORDER).reset_index()
signals = signals.rename(columns={'label': 'Model'})
publish(signals, 'readability_s04_signals')

,Model,Control (No Age),Implicit Cue (Adult),Explicit Age (Adult),Implicit Cue (Minor),Explicit Age (Minor)
0,GPT-5.6 Luna,9.63,10.07,9.45,9.27,7.59
1,Claude Haiku 4.5,8.54,8.72,8.37,7.92,6.26
2,Gemini 3.5 Flash Lite,9.61,9.82,9.55,9.20,8.20
3,DeepSeek-V4 Flash,8.46,8.30,7.87,7.90,6.27
4,Mistral Small 4,8.92,9.10,8.58,8.78,7.29
5,Gemma 4 31B,8.81,9.08,8.73,8.28,6.95


In [9]:
# R.6 Robustness. The floor sensitivity runs downward, since the measuring pass
# is at floor zero and a floor can be raised later but never lowered.
CUTS = [0, 25, 50, 75, 100]
sensitivity = []
for cut in CUTS:
    part = raw[raw['signal'].eq('stated')].copy()
    part = part[part['response_length'] >= cut]
    effects = [contrast_grades(part[part['label'] == label], 'fkgl',
                               STATED_MINOR, STATED_ADULT).mean()
               for label in ORDER]
    sensitivity.append({'Word Floor': cut,
                        'Replies Retained': int(len(part)),
                        'Macro-average (grades)': round(float(np.mean(effects)), 2),
                        'Models with the Majority Sign':
                            f'{sum(1 for e in effects if e < 0)} of 6'})
publish(pd.DataFrame(sensitivity), 'readability_s05_floor')


,Word Floor,Replies Retained,Macro-Average (Grades),Models with the Majority Sign
0,0,28642,-1.67,6 of 6
1,25,27809,-1.70,6 of 6
2,50,25896,-1.77,6 of 6
3,75,24166,-1.80,6 of 6
4,100,22198,-1.85,6 of 6


In [10]:
# The coarse mapping check, on the four ages where the two overlap. Run on the
# same cohort as Table G.3, so the comparison is between two mappings on one set
# of replies rather than between two sets, and scenario-weighted as that is.
OVERLAP = [7, 9, 11, 13]
coarse = frame[frame['targeted'] & frame['age'].isin(OVERLAP)].copy()
coarse['coarse_target'] = coarse['age'].map(language.coarse_target_grade)
coarse['coarse_aae'] = (coarse['fkgl'] - coarse['coarse_target']).abs()
mapping = by_scenario(coarse, ['aae', 'coarse_aae'], ['label']).round(2)
mapping['Difference'] = (mapping['coarse_aae'] - mapping['aae']).round(2)
mapping = mapping.reindex(ORDER).reset_index()
mapping.columns = ['Model', 'AAE, Continuous', 'AAE, Coarse', 'Difference']
publish(mapping, 'readability_s06_coarse')

,Model,"AAE, Continuous","AAE, Coarse",Difference
0,GPT-5.6 Luna,2.47,2.44,-0.03
1,Claude Haiku 4.5,1.73,1.63,-0.10
2,Gemini 3.5 Flash Lite,3.02,3.00,-0.02
3,DeepSeek-V4 Flash,1.90,1.79,-0.11
4,Mistral Small 4,2.54,2.49,-0.05
5,Gemma 4 31B,2.03,1.95,-0.08


In [11]:
# The eleven supplementary measures, reported once as a correlation matrix
# rather than as eleven results.
ALL = PRIMARY[:1] + SECONDARY + [
    'gunning_fog', 'ari', 'smog', 'p90_aoa', 'max_aoa', 'difficult_share',
    'aoa_coverage', 'sentence_length', 'word_length', 'ttr', 'mtld']
matrix = frame[ALL].corr().round(2)
matrix.index = [name.replace('_', ' ').title() for name in matrix.index]
matrix.columns = matrix.index
publish(matrix.reset_index().rename(columns={'index': 'Measure'}),
        'readability_s07_correlations')


,Measure,Fkgl,Fre,Mean Aoa,Response Length,Gunning Fog,Ari,Smog,P90 Aoa,Max Aoa,Difficult Share,Aoa Coverage,Sentence Length,Word Length,Ttr,Mtld
0,Fkgl,1.00,-0.93,0.72,-0.13,0.97,0.96,0.95,0.70,0.28,0.62,-0.36,0.60,0.70,0.30,0.36
1,Fre,-0.93,1.00,-0.87,0.14,-0.91,-0.90,-0.91,-0.82,-0.33,-0.73,0.44,-0.27,-0.89,-0.38,-0.52
2,Mean Aoa,0.72,-0.87,1.00,0.02,0.75,0.71,0.73,0.93,0.48,0.86,-0.37,0.02,0.86,0.22,0.54
3,Response Length,-0.13,0.14,0.02,1.00,-0.07,-0.17,-0.05,0.11,0.52,0.13,0.00,0.00,-0.12,-0.83,-0.14
4,Gunning Fog,0.97,-0.91,0.75,-0.07,1.00,0.93,0.97,0.74,0.31,0.66,-0.39,0.56,0.69,0.25,0.33
5,Ari,0.96,-0.90,0.71,-0.17,0.93,1.00,0.90,0.67,0.25,0.58,-0.36,0.57,0.75,0.37,0.42
6,Smog,0.95,-0.91,0.73,-0.05,0.97,0.90,1.00,0.74,0.33,0.63,-0.38,0.52,0.69,0.23,0.33
7,P90 Aoa,0.70,-0.82,0.93,0.11,0.74,0.67,0.74,1.00,0.51,0.84,-0.35,0.06,0.77,0.12,0.47
8,Max Aoa,0.28,-0.33,0.48,0.52,0.31,0.25,0.33,0.51,1.00,0.51,-0.18,0.05,0.32,-0.40,0.17
9,Difficult Share,0.62,-0.73,0.86,0.13,0.66,0.58,0.63,0.84,0.51,1.00,-0.34,0.03,0.68,0.09,0.41


In [12]:
# R.7 Does the reading-level effect survive when the outcome is held fixed?
#
# The check Section 4.3.1 admits it needs. The floor removes replies selected on
# the outcome and the outcome moves with the age, so the primary contrast could
# in principle be reporting that a model refuses more at young ages and that
# refusals are short, rather than that it writes more simply for a child.
#
# Two restrictions, and they do not carry equal weight.
#
# Benign scenarios carry the same benchmark-expected action at every age, so the
# expectation the floor selects against does not itself move with the treatment.
# A model can still refuse a benign request, and some do, so what the
# restriction gives is a stratum where the expected outcome is constant rather
# than one where the observed outcome is. That is the clean control of the two
# and the one to lead with.
#
# Total Compliance replies hold the outcome fixed directly rather than by
# choosing a stratum where it barely varies, but they condition on a
# post-treatment variable: selecting on an outcome the treatment affects can
# induce a difference where none exists. Reported as descriptive, with no test.
verdicts = pd.read_csv(RESULTS_DIR / 'classification.csv', dtype=str)
verdicts = verdicts[(verdicts['answer'] == 'Compliance')
                    & (verdicts['delivery_response'] == 'Yes')]
complied = set(zip(verdicts['prompt_id'], verdicts['model'],
                   verdicts['replicate'].astype(str)))

stated = frame[frame['signal'].eq('stated')].copy()
stated['complied'] = [
    (prompt, model, str(replicate)) in complied
    for prompt, model, replicate in zip(stated['prompt_id'], stated['model'],
                                        stated['replicate'])]

rows = []
for name, subset in [('Benign only',
                      stated[stated['scenario_type'] == 'Benign']),
                     ('Total Compliance only', stated[stated['complied']])]:
    for label in ORDER:
        diff = contrast_grades(subset[subset['label'] == label], 'fkgl',
                               STATED_MINOR, STATED_ADULT)
        if diff.size < 5:
            rows.append({'Restriction': name, 'Model': label,
                         'Effect (grades)': '', '95% CI Lower': '',
                         '95% CI Upper': '',
                         'Complete Scenarios': int(diff.size)})
            continue
        point, low, high = bootstrap_paired(diff)
        rows.append({'Restriction': name, 'Model': label,
                     'Effect (grades)': round(point, 2),
                     '95% CI Lower': round(low, 2),
                     '95% CI Upper': round(high, 2),
                     'Complete Scenarios': int(diff.size)})

outcome = pd.DataFrame(rows)
publish(outcome, 'readability_s08_outcome')

for name in outcome['Restriction'].unique():
    part = outcome[(outcome['Restriction'] == name)
                   & (outcome['Effect (grades)'] != '')]
    if len(part):
        print(f'{name:<24}{part["Effect (grades)"].mean():+.2f} grades on '
              f'average, {len(part)} of {len(ORDER)} models estimable')

Benign only             -1.76 grades on average, 6 of 6 models estimable
Total Compliance only   -1.90 grades on average, 6 of 6 models estimable


In [13]:
# The statutory boundary, seventeen against eighteen.
#
# The primary contrast compares six minor ages against two adult ones and cannot
# tell a gradual climb from a discontinuity at the boundary. These two ages are
# one year apart, so an effect here of the same order as the whole ladder would
# be consistent with a discontinuity at the adult boundary, and a small one with
# a gradual climb.
#
# Descriptive: an estimate and an interval, and no test. This was added as a
# diagnostic and opening a fresh six-test family at the end of the chapter is
# harder to defend than the diagnostic is worth. The register of
# Section 3.6.4 is unchanged by it.
rows = []
for label in ORDER:
    diff = contrast_grades(stated[stated['label'] == label], 'fkgl',
                           [17.0], [18.0])
    point, low, high = bootstrap_paired(diff)
    rows.append({'Model': label, 'Effect (grades)': round(point, 2),
                 '95% CI Lower': round(low, 2), '95% CI Upper': round(high, 2),
                 'Complete Scenarios': int(diff.size)})

boundary = pd.DataFrame(rows)
publish(boundary, 'readability_s09_boundary')

# The step against the whole ladder, which is what makes it readable.
for label in ORDER:
    levels = by_scenario(stated[stated['label'] == label], ['fkgl'],
                         ['age'])['fkgl']
    step = levels.get(18.0) - levels.get(17.0)
    span = levels.get(21.0) - levels.get(7.0)
    print(f'{label:<24}17 to 18 {step:+.2f} of {span:+.2f} across the ladder '
          f'({step / span:.0%})')

GPT-5.6 Luna            17 to 18 +0.31 of +3.47 across the ladder (9%)
Claude Haiku 4.5        17 to 18 +0.40 of +3.88 across the ladder (10%)
Gemini 3.5 Flash Lite   17 to 18 +0.14 of +3.00 across the ladder (5%)
DeepSeek-V4 Flash       17 to 18 +0.10 of +3.00 across the ladder (3%)
Mistral Small 4         17 to 18 +0.19 of +2.48 across the ladder (8%)
Gemma 4 31B             17 to 18 +0.19 of +3.63 across the ladder (5%)


In [14]:
# The upper tail of the vocabulary, supplementary and descriptive.
#
# Mean AoA is dominated by the common early-acquired words every reply is mostly
# made of, so a handful of words well beyond a young reader moves it very
# little. P90 AoA is the ninetieth percentile of the same distribution and is
# what shows the difficult end of it. No test, and no promotion into the measure
# hierarchy of Section 3.5.2, which is fixed.
#
# Over all returned replies at a stated minor age, with no fifty-word floor
# applied, and not over the smaller cohort the calibration table uses. The floor exists because the five readability formulas are
# unstable on a short text; the vocabulary measures are defined at any length,
# so applying it here would discard the short replies for no reason.
minor_lexical = frame[frame['signal'].eq('stated')
                      & frame['age'].isin(STATED_MINOR)]
LEXICAL = ['mean_aoa', 'p90_aoa', 'max_aoa', 'difficult_share', 'aoa_coverage']
tail = by_scenario(minor_lexical, LEXICAL, ['label'])
tail = tail.round(3).reindex(ORDER).reset_index()
tail.columns = ['Model', 'Mean AoA', 'P90 AoA', 'Max AoA', 'Difficult Share',
                'AoA Coverage']
publish(tail, 'readability_s10_lexical_tail')

print(f'{len(minor_lexical):,} replies at a stated minor age, no floor applied; '
      f'the calibration table uses {len(targeted):,} that also cleared it')

21,442 replies at a stated minor age, no floor applied; the calibration table uses 19,340 that also cleared it


In [15]:
write_captions()
print(', '.join(f'{count} {kind}' for kind, count
                in sorted(analysis.WRITTEN.items())) + ' written')


described in captions.yml but not written: readability_conditioning
2 main table, 10 supplement table written
